# Cleaning

## Importing Libraries and data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
import re
import os
from tqdm.notebook import tqdm

#from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans,MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

In [2]:
#os.getcwd()
#df_2009 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
#df_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")
#df = pd.concat([df_2009, df_2010], ignore_index=True)

In [3]:
#df.to_csv("../data/raw/online_retail.csv", index=False)
df = pd.read_csv("../data/raw/online_retail.csv")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


## Data Type Conversion and Text Standardization

In [5]:
df['Invoice'] = df['Invoice'].astype(str).str.upper().str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Customer ID'] = df['Customer ID'].astype('Int64')
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()
df['Description'] = df['Description'].astype(str).str.upper().str.strip()
df['Country'] = df['Country'].astype(str).str.upper().str.strip()

In [6]:
# delete rows where Customer ID is missing
df = df[df['Customer ID'].notna()].copy()

## Resolving conflicts between columns

In [7]:
###  Checking customer country mismatch

# Working only with known customer
df_known_customer = df[df['Customer ID'].notna()].copy()

# Count unique Country per Customer ID
customer_country_check = (
    df_known_customer
    .groupby('Customer ID')
    .agg(
        n_countries=('Country', lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

# Customers linked to more than one country
country_mismatch = customer_country_check[
    customer_country_check['n_countries'] > 1
]

# Save all original rows belonging to those multiple country customers
customer_country_exceptions = (
    df_known_customer[
        df_known_customer['Customer ID'].isin(country_mismatch['Customer ID'])
    ]
    .sort_values(['Customer ID', 'Country', 'InvoiceDate'])
)

if country_mismatch.shape[0] == 0:
    print('No mismatch between customer ID and Country')
else:
    print(f'{country_mismatch.shape[0]} customers order products from more than one Country. ID\'s of the customres are:')
    print(customer_country_exceptions[
    ['Customer ID', 'Country']
].drop_duplicates().sort_values(['Customer ID', 'Country']).head(50))

13 customers order products from more than one Country. ID's of the customres are:
        Customer ID         Country
85911         12370         AUSTRIA
555193        12370          CYPRUS
703342        12394         BELGIUM
908550        12394         DENMARK
572318        12413          FRANCE
428175        12413           SPAIN
12491         12417         BELGIUM
365827        12417           SPAIN
39830         12422       AUSTRALIA
205288        12422     SWITZERLAND
467038        12423         BELGIUM
321986        12423         DENMARK
693860        12429         AUSTRIA
233034        12429         DENMARK
110890        12431       AUSTRALIA
57311         12431         BELGIUM
739771        12449         BELGIUM
356635        12449         DENMARK
39735         12455          CYPRUS
737728        12455           SPAIN
872010        12457          CYPRUS
365567        12457     SWITZERLAND
580184        12652          FRANCE
367186        12652         GERMANY
70977         127

In [8]:
### Removing mismatch between country and customer

# Work only with known customers
df_known_customer = df[df['Customer ID'].notna()].copy()

# Invoice count and total quantity by Customer ID
customer_country_stats = (
    df_known_customer
    .groupby(['Customer ID', 'Country'])
    .agg(
        n_invoices=('Invoice', 'nunique'),
        total_quantity=('Quantity', 'sum')
    )
    .reset_index()
)

# Pick mode country:
# 1. highest number of invoices
# 2. if tied, highest total quantity
mode_country_per_customer = (
    customer_country_stats
    .sort_values(
        ['Customer ID', 'n_invoices', 'total_quantity'],
        ascending=[True, False, False]
    )
    .drop_duplicates('Customer ID', keep='first')
    [['Customer ID', 'Country']]
    .rename(columns={'Country': 'mode_country'})
)

# Add mode country to original df
df = df.merge(
    mode_country_per_customer,
    on='Customer ID',
    how='left'
)

# Replace Country only for rows with known Customer ID
df.loc[df['Customer ID'].notna(), 'Country'] = df.loc[
    df['Customer ID'].notna(),
    'mode_country'
]

# Drop helper column
df = df.drop(columns='mode_country')

print('Mismatch removed between customer ID and Country, by updating country column')

Mismatch removed between customer ID and Country, by updating country column


In [9]:
### Chacking whether rows corresponds to a same invoice referes to a same customer id 

# Count unique Customer ID  per Invoice
# dropna=False means missing Customer ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer ID'])
)

#invoice_exceptions.to_csv("same_invoice_customer_country_exceptions.csv", index=False)

if bad_invoices.shape[0] == 0:
    print('No mismatch between invoice and customer ID')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and customer ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer ID', 'Country']].drop_duplicates().head(5))

No mismatch between invoice and customer ID


In [10]:
### Updating description column
##  Same stock code should point to same items.

collapsed = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .rename('count')
      .reset_index()
)

collapsed['rn'] = (
    collapsed.groupby('StockCode')['count']
             .rank(method='first', ascending=False)
)

collapsed = collapsed[collapsed['rn'] == 1]

df = df.merge(collapsed, on='StockCode', how='left', suffixes=('', '_top'))
df['Description'] = df['Description_top']
df = df.drop(columns=['Description_top', 'rn','count'])

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 824364 entries, 0 to 824363
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      824364 non-null  str           
 1   StockCode    824364 non-null  str           
 2   Description  824364 non-null  str           
 3   Quantity     824364 non-null  int64         
 4   InvoiceDate  824364 non-null  datetime64[us]
 5   Price        824364 non-null  float64       
 6   Customer ID  824364 non-null  Int64         
 7   Country      824364 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 51.1 MB


# Handling Return Order

In [12]:
### finding cancelled rows

df['status'] = pd.NA

negative_rows = df[df['Quantity'] < 0].sort_values('InvoiceDate')

for neg_i in tqdm(negative_rows.index, total=len(negative_rows)):
    
    neg_row = df.loc[neg_i]
    
    candidates = df[
        (df['Quantity'] > 0) &
        (df['status'].isna()) &
        (df['Customer ID'] == neg_row['Customer ID']) &
        (df['StockCode'] == neg_row['StockCode']) &
        (df['InvoiceDate'] <= neg_row['InvoiceDate']) &
        (df['Quantity'] == -neg_row['Quantity']) &
        (df['Price'] == neg_row['Price'])
    ]
    
    if len(candidates) > 0:
        match_i = candidates['InvoiceDate'].idxmax()
        df.loc[match_i, 'status'] = 'cancelled'
        df.loc[neg_i, 'status'] = 'return: purchase found'
    else:
        df.loc[neg_i, 'status'] = 'return: purchase not found'

df.loc[
    (df['Quantity'] > 0) & (df['status'].isna()),
    'status'
] = 'completed'


  0%|          | 0/18744 [00:00<?, ?it/s]

In [13]:
## sorting df based on status and date time
status_order = {
    'completed': 1,
    'return: purchase found': 2,
    'cancelled': 3,
    'return: purchase not found': 4
}

df = (
    df.assign(
        status_order=df['status'].map(status_order)
    )
    .sort_values(
        ['status_order', 'InvoiceDate'],
        ascending=[True, True]
    )
    .drop(columns='status_order')
)

In [14]:
if 'line_id' not in df.columns:
    df.insert(0, 'line_id', range(1, len(df) + 1))

In [15]:
## creating dataframe for saving
df_cleaned = df[df['status']=='completed']
df_return = df[df['status'] == 'return: purchase found']

# droping status column before saving to csv
df_cleaned.drop(columns = 'status', inplace=True)
df_return.drop(columns = 'status', inplace=True)

## Saving to csv
df_cleaned.to_csv('clean_purchase.csv', index=False)
df_return.to_csv('return_order.csv', index=False)

In [16]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 799406 entries, 0 to 824363
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   line_id      799406 non-null  int64         
 1   Invoice      799406 non-null  str           
 2   StockCode    799406 non-null  str           
 3   Description  799406 non-null  str           
 4   Quantity     799406 non-null  int64         
 5   InvoiceDate  799406 non-null  datetime64[us]
 6   Price        799406 non-null  float64       
 7   Customer ID  799406 non-null  Int64         
 8   Country      799406 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 61.8 MB


In [17]:
df_return.info()

<class 'pandas.DataFrame'>
Index: 6214 entries, 1292 to 823884
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   line_id      6214 non-null   int64         
 1   Invoice      6214 non-null   str           
 2   StockCode    6214 non-null   str           
 3   Description  6214 non-null   str           
 4   Quantity     6214 non-null   int64         
 5   InvoiceDate  6214 non-null   datetime64[us]
 6   Price        6214 non-null   float64       
 7   Customer ID  6214 non-null   Int64         
 8   Country      6214 non-null   str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 491.5 KB
